# CSE 472 Assignment 2: Decision Trees, Random Forests, and Extra Trees

**Implementation of Tree-Based Learning Algorithms from Scratch**

This notebook contains:
1. Custom implementations of Decision Tree, Random Forest, and Extra Trees
2. Evaluation on Iris and Wine datasets
3. Comparison with scikit-learn implementations
4. Comprehensive performance analysis

---

**ID: 2005076** <br>
**Name: S.M Kausar Parvej**

---

## 1. Import Libraries and Setup

In [2]:
import numpy as np
import pandas as pd
from collections import Counter
from sklearn.datasets import load_iris, load_wine
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.preprocessing import label_binarize
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
RANDOM_SEED = 76
np.random.seed(RANDOM_SEED)

## 2. Load and Prepare Datasets

In [3]:
# Load datasets
iris = load_iris()
wine = load_wine()

# Prepare Iris dataset
X_iris, y_iris = iris.data, iris.target
X_iris_train, X_iris_test, y_iris_train, y_iris_test = train_test_split(
    X_iris, y_iris, test_size=0.3, random_state=RANDOM_SEED, stratify=y_iris
)

# Prepare Wine dataset
X_wine, y_wine = wine.data, wine.target
X_wine_train, X_wine_test, y_wine_train, y_wine_test = train_test_split(
    X_wine, y_wine, test_size=0.3, random_state=RANDOM_SEED, stratify=y_wine
)

print("Dataset Information:")
print(f"Iris - Train: {X_iris_train.shape}, Test: {X_iris_test.shape}")
print(f"Wine - Train: {X_wine_train.shape}, Test: {X_wine_test.shape}")

Dataset Information:
Iris - Train: (105, 4), Test: (45, 4)
Wine - Train: (124, 13), Test: (54, 13)


## 3. Custom Decision Tree Implementation

In [4]:
class Node:
    """Node class for decision tree"""
    def __init__(self, feature=None, threshold=None, left=None, right=None, value=None):
        self.feature = feature        # Feature index to split on
        self.threshold = threshold    # Threshold value for split
        self.left = left             # Left child node
        self.right = right           # Right child node
        self.value = value           # Class value if leaf node


class CustomDecisionTree:
    """Custom Decision Tree Classifier implementation"""

    def __init__(self, max_depth=None, min_samples_split=2, criterion='gini'):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.criterion = criterion
        self.root = None

    def _gini(self, y):
        """Calculate Gini impurity"""
        proportions = np.bincount(y) / len(y)
        return 1 - np.sum(proportions ** 2)

    def _entropy(self, y):
        """Calculate entropy"""
        proportions = np.bincount(y) / len(y)
        return -np.sum([p * np.log2(p) for p in proportions if p > 0])

    def _calculate_impurity(self, y):
        """Calculate impurity based on criterion"""
        if self.criterion == 'gini':
            return self._gini(y)
        else:
            return self._entropy(y)

    def _information_gain(self, X_column, y, threshold):
        """Calculate information gain for a split"""
        parent_impurity = self._calculate_impurity(y)

        left_mask = X_column <= threshold
        right_mask = ~left_mask

        if np.sum(left_mask) == 0 or np.sum(right_mask) == 0:
            return 0

        n = len(y)
        n_left, n_right = np.sum(left_mask), np.sum(right_mask)
        left_impurity = self._calculate_impurity(y[left_mask])
        right_impurity = self._calculate_impurity(y[right_mask])

        child_impurity = (n_left / n) * left_impurity + (n_right / n) * right_impurity
        return parent_impurity - child_impurity

    def _best_split(self, X, y):
        """Find the best feature and threshold to split on"""
        best_gain = -1
        best_feature = None
        best_threshold = None

        n_features = X.shape[1]

        for feature_idx in range(n_features):
            X_column = X[:, feature_idx]
            thresholds = np.unique(X_column)

            for threshold in thresholds:
                gain = self._information_gain(X_column, y, threshold)

                if gain > best_gain:
                    best_gain = gain
                    best_feature = feature_idx
                    best_threshold = threshold

        return best_feature, best_threshold

    def _build_tree(self, X, y, depth=0):
        """Recursively build the decision tree"""
        n_samples, n_features = X.shape

        # Handle empty dataset
        if n_samples == 0 or len(y) == 0:
            return Node(value=0)  # Return default class

        n_classes = len(np.unique(y))

        # Stopping criteria
        if (self.max_depth is not None and depth >= self.max_depth) or \
           n_classes == 1 or \
           n_samples < self.min_samples_split:
            leaf_value = Counter(y).most_common(1)[0][0]
            return Node(value=leaf_value)

        # Find best split
        best_feature, best_threshold = self._best_split(X, y)

        if best_feature is None:
            leaf_value = Counter(y).most_common(1)[0][0]
            return Node(value=leaf_value)

        # Split the data
        left_mask = X[:, best_feature] <= best_threshold
        right_mask = ~left_mask

        # Ensure both splits have data
        if np.sum(left_mask) == 0 or np.sum(right_mask) == 0:
            leaf_value = Counter(y).most_common(1)[0][0]
            return Node(value=leaf_value)

        # Build left and right subtrees
        left = self._build_tree(X[left_mask], y[left_mask], depth + 1)
        right = self._build_tree(X[right_mask], y[right_mask], depth + 1)

        return Node(feature=best_feature, threshold=best_threshold, left=left, right=right)

    def fit(self, X, y):
        """Fit the decision tree"""
        self.root = self._build_tree(X, y)
        return self

    def _predict_sample(self, x, node):
        """Predict a single sample"""
        if node.value is not None:
            return node.value

        if x[node.feature] <= node.threshold:
            return self._predict_sample(x, node.left)
        else:
            return self._predict_sample(x, node.right)

    def predict(self, X):
        """Predict class labels for samples in X"""
        return np.array([self._predict_sample(x, self.root) for x in X])

    def predict_proba(self, X):
        """Predict class probabilities for samples in X"""
        # For simplicity, return one-hot encoding of predictions
        predictions = self.predict(X)
        n_classes = len(np.unique(predictions))
        proba = np.zeros((len(X), n_classes))
        for i, pred in enumerate(predictions):
            proba[i, pred] = 1.0
        return proba

print("Custom Decision Tree implementation completed!")

Custom Decision Tree implementation completed!


## 4. Custom Random Forest Implementation

In [5]:
class CustomRandomForest:
    """Custom Random Forest Classifier implementation"""

    def __init__(self, n_estimators=100, max_depth=None, min_samples_split=2,
                 max_features='sqrt', criterion='gini', random_state=None):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.max_features = max_features
        self.criterion = criterion
        self.random_state = random_state
        self.trees = []

    def _bootstrap_sample(self, X, y):
        """Create a bootstrap sample from the dataset"""
        n_samples = X.shape[0]
        indices = np.random.choice(n_samples, n_samples, replace=True)
        return X[indices], y[indices]

    def _get_n_features(self, n_total_features):
        """Determine number of features to consider at each split"""
        if self.max_features == 'sqrt':
            return int(np.sqrt(n_total_features))
        elif self.max_features == 'log2':
            return int(np.log2(n_total_features))
        elif isinstance(self.max_features, int):
            return self.max_features
        else:
            return n_total_features

    def fit(self, X, y):
        """Fit the random forest"""
        if self.random_state is not None:
            np.random.seed(self.random_state)

        self.trees = []
        self.n_classes = len(np.unique(y))

        for _ in range(self.n_estimators):
            # Create bootstrap sample
            X_sample, y_sample = self._bootstrap_sample(X, y)

            # Create a modified decision tree with random feature selection
            tree = RandomFeatureDecisionTree(
                max_depth=self.max_depth,
                min_samples_split=self.min_samples_split,
                criterion=self.criterion,
                max_features=self._get_n_features(X.shape[1])
            )
            tree.fit(X_sample, y_sample)
            self.trees.append(tree)

        return self

    def predict(self, X):
        """Predict class labels using majority voting"""
        predictions = np.array([tree.predict(X) for tree in self.trees])
        # Majority vote
        return np.array([Counter(predictions[:, i]).most_common(1)[0][0]
                        for i in range(X.shape[0])])

    def predict_proba(self, X):
        """Predict class probabilities by averaging tree predictions"""
        all_probas = np.array([tree.predict_proba(X) for tree in self.trees])
        return np.mean(all_probas, axis=0)


class RandomFeatureDecisionTree(CustomDecisionTree):
    """Decision Tree with random feature selection for Random Forest"""

    def __init__(self, max_depth=None, min_samples_split=2, criterion='gini', max_features=None):
        super().__init__(max_depth, min_samples_split, criterion)
        self.max_features = max_features

    def _best_split(self, X, y):
        """Find best split considering only a random subset of features"""
        best_gain = -1
        best_feature = None
        best_threshold = None

        n_features = X.shape[1]

        # Randomly select features to consider
        if self.max_features is not None and self.max_features < n_features:
            feature_indices = np.random.choice(n_features, self.max_features, replace=False)
        else:
            feature_indices = range(n_features)

        for feature_idx in feature_indices:
            X_column = X[:, feature_idx]
            thresholds = np.unique(X_column)

            for threshold in thresholds:
                gain = self._information_gain(X_column, y, threshold)

                if gain > best_gain:
                    best_gain = gain
                    best_feature = feature_idx
                    best_threshold = threshold

        return best_feature, best_threshold

print("Custom Random Forest implementation completed!")

Custom Random Forest implementation completed!


## 5. Custom Extra Trees Implementation

In [6]:
class CustomExtraTrees:
    """Custom Extra Trees (Extremely Randomized Trees) Classifier implementation"""

    def __init__(self, n_estimators=100, max_depth=None, min_samples_split=2,
                 max_features='sqrt', criterion='gini', random_state=None):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.max_features = max_features
        self.criterion = criterion
        self.random_state = random_state
        self.trees = []

    def _get_n_features(self, n_total_features):
        """Determine number of features to consider at each split"""
        if self.max_features == 'sqrt':
            return int(np.sqrt(n_total_features))
        elif self.max_features == 'log2':
            return int(np.log2(n_total_features))
        elif isinstance(self.max_features, int):
            return self.max_features
        else:
            return n_total_features

    def fit(self, X, y):
        """Fit the extra trees ensemble"""
        if self.random_state is not None:
            np.random.seed(self.random_state)

        self.trees = []
        self.n_classes = len(np.unique(y))

        for _ in range(self.n_estimators):
            # Extra Trees uses the full dataset (no bootstrap)
            tree = ExtraDecisionTree(
                max_depth=self.max_depth,
                min_samples_split=self.min_samples_split,
                criterion=self.criterion,
                max_features=self._get_n_features(X.shape[1])
            )
            tree.fit(X, y)
            self.trees.append(tree)

        return self

    def predict(self, X):
        """Predict class labels using majority voting"""
        predictions = np.array([tree.predict(X) for tree in self.trees])
        return np.array([Counter(predictions[:, i]).most_common(1)[0][0]
                        for i in range(X.shape[0])])

    def predict_proba(self, X):
        """Predict class probabilities by averaging tree predictions"""
        all_probas = np.array([tree.predict_proba(X) for tree in self.trees])
        return np.mean(all_probas, axis=0)


class ExtraDecisionTree(CustomDecisionTree):
    """Decision Tree with random threshold selection for Extra Trees"""

    def __init__(self, max_depth=None, min_samples_split=2, criterion='gini', max_features=None):
        super().__init__(max_depth, min_samples_split, criterion)
        self.max_features = max_features

    def _best_split(self, X, y):
        """Find best split using random thresholds and random features"""
        best_gain = -1
        best_feature = None
        best_threshold = None

        n_features = X.shape[1]

        # Randomly select features to consider
        if self.max_features is not None and self.max_features < n_features:
            feature_indices = np.random.choice(n_features, self.max_features, replace=False)
        else:
            feature_indices = range(n_features)

        for feature_idx in feature_indices:
            X_column = X[:, feature_idx]

            # Key difference: randomly select threshold instead of trying all
            min_val, max_val = X_column.min(), X_column.max()
            if min_val == max_val:
                continue

            threshold = np.random.uniform(min_val, max_val)
            gain = self._information_gain(X_column, y, threshold)

            if gain > best_gain:
                best_gain = gain
                best_feature = feature_idx
                best_threshold = threshold

        return best_feature, best_threshold

print("Custom Extra Trees implementation completed!")

Custom Extra Trees implementation completed!


## 6. Evaluation Functions

In [7]:
def evaluate_model(model, X_train, y_train, X_test, y_test, model_name, dataset_name):
    """
    Evaluate a model and return performance metrics

    Returns: dict with accuracy, f1_score, and roc_auc
    """
    # Train the model
    model.fit(X_train, y_train)

    # Make predictions
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)

    # Calculate metrics
    accuracy = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='macro')

    # Calculate AUROC for multiclass
    n_classes = len(np.unique(y_test))
    y_test_bin = label_binarize(y_test, classes=range(n_classes))

    # Handle probability matrix shape
    if y_proba.shape[1] < n_classes:
        # Pad with zeros if needed
        y_proba_padded = np.zeros((y_proba.shape[0], n_classes))
        y_proba_padded[:, :y_proba.shape[1]] = y_proba
        y_proba = y_proba_padded

    roc_auc = roc_auc_score(y_test_bin, y_proba, average='macro', multi_class='ovr')

    return {
        'Model': model_name,
        'Dataset': dataset_name,
        'Accuracy': accuracy,
        'F1-Score': f1,
        'AUROC': roc_auc
    }

print("Evaluation functions defined!")

Evaluation functions defined!


## 7. Train and Evaluate Custom Models

In [8]:
# Hyperparameters
MAX_DEPTH = 10
MIN_SAMPLES_SPLIT = 2
N_ESTIMATORS = 100
MAX_FEATURES = 'sqrt'

results = []

print("Training Custom Models...")
print("=" * 60)

# Custom Decision Tree - Iris
print("\n1. Custom Decision Tree on Iris dataset...")
custom_dt_iris = CustomDecisionTree(max_depth=MAX_DEPTH, min_samples_split=MIN_SAMPLES_SPLIT)
result = evaluate_model(custom_dt_iris, X_iris_train, y_iris_train, X_iris_test, y_iris_test,
                        'Custom Decision Tree', 'Iris')
results.append(result)
print(f"   Accuracy: {result['Accuracy']:.4f}, F1: {result['F1-Score']:.4f}, AUROC: {result['AUROC']:.4f}")

# Custom Decision Tree - Wine
print("2. Custom Decision Tree on Wine dataset...")
custom_dt_wine = CustomDecisionTree(max_depth=MAX_DEPTH, min_samples_split=MIN_SAMPLES_SPLIT)
result = evaluate_model(custom_dt_wine, X_wine_train, y_wine_train, X_wine_test, y_wine_test,
                        'Custom Decision Tree', 'Wine')
results.append(result)
print(f"   Accuracy: {result['Accuracy']:.4f}, F1: {result['F1-Score']:.4f}, AUROC: {result['AUROC']:.4f}")

# Custom Random Forest - Iris
print("3. Custom Random Forest on Iris dataset...")
custom_rf_iris = CustomRandomForest(n_estimators=N_ESTIMATORS, max_depth=MAX_DEPTH,
                                    min_samples_split=MIN_SAMPLES_SPLIT, max_features=MAX_FEATURES,
                                    random_state=RANDOM_SEED)
result = evaluate_model(custom_rf_iris, X_iris_train, y_iris_train, X_iris_test, y_iris_test,
                        'Custom Random Forest', 'Iris')
results.append(result)
print(f"   Accuracy: {result['Accuracy']:.4f}, F1: {result['F1-Score']:.4f}, AUROC: {result['AUROC']:.4f}")

# Custom Random Forest - Wine
print("4. Custom Random Forest on Wine dataset...")
custom_rf_wine = CustomRandomForest(n_estimators=N_ESTIMATORS, max_depth=MAX_DEPTH,
                                    min_samples_split=MIN_SAMPLES_SPLIT, max_features=MAX_FEATURES,
                                    random_state=RANDOM_SEED)
result = evaluate_model(custom_rf_wine, X_wine_train, y_wine_train, X_wine_test, y_wine_test,
                        'Custom Random Forest', 'Wine')
results.append(result)
print(f"   Accuracy: {result['Accuracy']:.4f}, F1: {result['F1-Score']:.4f}, AUROC: {result['AUROC']:.4f}")

# Custom Extra Trees - Iris
print("5. Custom Extra Trees on Iris dataset...")
custom_et_iris = CustomExtraTrees(n_estimators=N_ESTIMATORS, max_depth=MAX_DEPTH,
                                  min_samples_split=MIN_SAMPLES_SPLIT, max_features=MAX_FEATURES,
                                  random_state=RANDOM_SEED)
result = evaluate_model(custom_et_iris, X_iris_train, y_iris_train, X_iris_test, y_iris_test,
                        'Custom Extra Trees', 'Iris')
results.append(result)
print(f"   Accuracy: {result['Accuracy']:.4f}, F1: {result['F1-Score']:.4f}, AUROC: {result['AUROC']:.4f}")

# Custom Extra Trees - Wine
print("6. Custom Extra Trees on Wine dataset...")
custom_et_wine = CustomExtraTrees(n_estimators=N_ESTIMATORS, max_depth=MAX_DEPTH,
                                  min_samples_split=MIN_SAMPLES_SPLIT, max_features=MAX_FEATURES,
                                  random_state=RANDOM_SEED)
result = evaluate_model(custom_et_wine, X_wine_train, y_wine_train, X_wine_test, y_wine_test,
                        'Custom Extra Trees', 'Wine')
results.append(result)
print(f"   Accuracy: {result['Accuracy']:.4f}, F1: {result['F1-Score']:.4f}, AUROC: {result['AUROC']:.4f}")

print("\n" + "=" * 60)
print("Custom models training completed!")

Training Custom Models...

1. Custom Decision Tree on Iris dataset...
   Accuracy: 0.8667, F1: 0.8667, AUROC: 0.9000
2. Custom Decision Tree on Wine dataset...
   Accuracy: 0.8889, F1: 0.8882, AUROC: 0.9221
3. Custom Random Forest on Iris dataset...
   Accuracy: 0.9111, F1: 0.9118, AUROC: 0.9844
4. Custom Random Forest on Wine dataset...
   Accuracy: 0.9815, F1: 0.9811, AUROC: 0.9998
5. Custom Extra Trees on Iris dataset...
   Accuracy: 0.9333, F1: 0.9333, AUROC: 0.9926
6. Custom Extra Trees on Wine dataset...
   Accuracy: 0.9815, F1: 0.9811, AUROC: 1.0000

Custom models training completed!


## 8. Train and Evaluate Scikit-learn Models

In [9]:
print("Training Scikit-learn Models...")
print("=" * 60)

# Sklearn Decision Tree - Iris
print("\n1. Sklearn Decision Tree on Iris dataset...")
sklearn_dt_iris = DecisionTreeClassifier(max_depth=MAX_DEPTH, min_samples_split=MIN_SAMPLES_SPLIT,
                                         random_state=RANDOM_SEED)
result = evaluate_model(sklearn_dt_iris, X_iris_train, y_iris_train, X_iris_test, y_iris_test,
                        'Sklearn Decision Tree', 'Iris')
results.append(result)
print(f"   Accuracy: {result['Accuracy']:.4f}, F1: {result['F1-Score']:.4f}, AUROC: {result['AUROC']:.4f}")

# Sklearn Decision Tree - Wine
print("2. Sklearn Decision Tree on Wine dataset...")
sklearn_dt_wine = DecisionTreeClassifier(max_depth=MAX_DEPTH, min_samples_split=MIN_SAMPLES_SPLIT,
                                        random_state=RANDOM_SEED)
result = evaluate_model(sklearn_dt_wine, X_wine_train, y_wine_train, X_wine_test, y_wine_test,
                        'Sklearn Decision Tree', 'Wine')
results.append(result)
print(f"   Accuracy: {result['Accuracy']:.4f}, F1: {result['F1-Score']:.4f}, AUROC: {result['AUROC']:.4f}")

# Sklearn Random Forest - Iris
print("3. Sklearn Random Forest on Iris dataset...")
sklearn_rf_iris = RandomForestClassifier(n_estimators=N_ESTIMATORS, max_depth=MAX_DEPTH,
                                        min_samples_split=MIN_SAMPLES_SPLIT, max_features=MAX_FEATURES,
                                        random_state=RANDOM_SEED)
result = evaluate_model(sklearn_rf_iris, X_iris_train, y_iris_train, X_iris_test, y_iris_test,
                        'Sklearn Random Forest', 'Iris')
results.append(result)
print(f"   Accuracy: {result['Accuracy']:.4f}, F1: {result['F1-Score']:.4f}, AUROC: {result['AUROC']:.4f}")

# Sklearn Random Forest - Wine
print("4. Sklearn Random Forest on Wine dataset...")
sklearn_rf_wine = RandomForestClassifier(n_estimators=N_ESTIMATORS, max_depth=MAX_DEPTH,
                                        min_samples_split=MIN_SAMPLES_SPLIT, max_features=MAX_FEATURES,
                                        random_state=RANDOM_SEED)
result = evaluate_model(sklearn_rf_wine, X_wine_train, y_wine_train, X_wine_test, y_wine_test,
                        'Sklearn Random Forest', 'Wine')
results.append(result)
print(f"   Accuracy: {result['Accuracy']:.4f}, F1: {result['F1-Score']:.4f}, AUROC: {result['AUROC']:.4f}")

# Sklearn Extra Trees - Iris
print("5. Sklearn Extra Trees on Iris dataset...")
sklearn_et_iris = ExtraTreesClassifier(n_estimators=N_ESTIMATORS, max_depth=MAX_DEPTH,
                                      min_samples_split=MIN_SAMPLES_SPLIT, max_features=MAX_FEATURES,
                                      random_state=RANDOM_SEED)
result = evaluate_model(sklearn_et_iris, X_iris_train, y_iris_train, X_iris_test, y_iris_test,
                        'Sklearn Extra Trees', 'Iris')
results.append(result)
print(f"   Accuracy: {result['Accuracy']:.4f}, F1: {result['F1-Score']:.4f}, AUROC: {result['AUROC']:.4f}")

# Sklearn Extra Trees - Wine
print("6. Sklearn Extra Trees on Wine dataset...")
sklearn_et_wine = ExtraTreesClassifier(n_estimators=N_ESTIMATORS, max_depth=MAX_DEPTH,
                                      min_samples_split=MIN_SAMPLES_SPLIT, max_features=MAX_FEATURES,
                                      random_state=RANDOM_SEED)
result = evaluate_model(sklearn_et_wine, X_wine_train, y_wine_train, X_wine_test, y_wine_test,
                        'Sklearn Extra Trees', 'Wine')
results.append(result)
print(f"   Accuracy: {result['Accuracy']:.4f}, F1: {result['F1-Score']:.4f}, AUROC: {result['AUROC']:.4f}")

print("\n" + "=" * 60)
print("Scikit-learn models training completed!")

Training Scikit-learn Models...

1. Sklearn Decision Tree on Iris dataset...
   Accuracy: 0.8667, F1: 0.8667, AUROC: 0.9000
2. Sklearn Decision Tree on Wine dataset...
   Accuracy: 0.9074, F1: 0.9066, AUROC: 0.9346
3. Sklearn Random Forest on Iris dataset...
   Accuracy: 0.9556, F1: 0.9556, AUROC: 0.9926
4. Sklearn Random Forest on Wine dataset...
   Accuracy: 0.9630, F1: 0.9625, AUROC: 1.0000
5. Sklearn Extra Trees on Iris dataset...
   Accuracy: 0.9333, F1: 0.9333, AUROC: 0.9926
6. Sklearn Extra Trees on Wine dataset...
   Accuracy: 0.9815, F1: 0.9811, AUROC: 1.0000

Scikit-learn models training completed!


## 9. Results Comparison and Analysis

In [10]:
# Create results DataFrame
results_df = pd.DataFrame(results)

print("\n" + "=" * 80)
print("COMPLETE RESULTS TABLE")
print("=" * 80)
print(results_df.to_string(index=False))
print("=" * 80)

# Separate results by dataset for clearer comparison
print("\n\n" + "=" * 80)
print("IRIS DATASET RESULTS")
print("=" * 80)
iris_results = results_df[results_df['Dataset'] == 'Iris'].copy()
iris_results = iris_results.drop('Dataset', axis=1)
print(iris_results.to_string(index=False))
print("=" * 80)

print("\n\n" + "=" * 80)
print("WINE DATASET RESULTS")
print("=" * 80)
wine_results = results_df[results_df['Dataset'] == 'Wine'].copy()
wine_results = wine_results.drop('Dataset', axis=1)
print(wine_results.to_string(index=False))
print("=" * 80)


COMPLETE RESULTS TABLE
                Model Dataset  Accuracy  F1-Score    AUROC
 Custom Decision Tree    Iris  0.866667  0.866667 0.900000
 Custom Decision Tree    Wine  0.888889  0.888158 0.922076
 Custom Random Forest    Iris  0.911111  0.911803 0.984444
 Custom Random Forest    Wine  0.981481  0.981117 0.999759
   Custom Extra Trees    Iris  0.933333  0.933259 0.992593
   Custom Extra Trees    Wine  0.981481  0.981117 1.000000
Sklearn Decision Tree    Iris  0.866667  0.866667 0.900000
Sklearn Decision Tree    Wine  0.907407  0.906589 0.934642
Sklearn Random Forest    Iris  0.955556  0.955556 0.992593
Sklearn Random Forest    Wine  0.962963  0.962500 1.000000
  Sklearn Extra Trees    Iris  0.933333  0.933259 0.992593
  Sklearn Extra Trees    Wine  0.981481  0.981117 1.000000


IRIS DATASET RESULTS
                Model  Accuracy  F1-Score    AUROC
 Custom Decision Tree  0.866667  0.866667 0.900000
 Custom Random Forest  0.911111  0.911803 0.984444
   Custom Extra Trees  0.933333  

### Analysis Summary

In [11]:
print("\n" + "=" * 80)
print("KEY FINDINGS AND ANALYSIS")
print("=" * 80)

# Analysis 1: Ensemble vs Single Tree
print("\n1. ENSEMBLE METHODS vs DECISION TREES:")
print("-" * 80)
for dataset in ['Iris', 'Wine']:
    dataset_results = results_df[results_df['Dataset'] == dataset]

    print(f"\n{dataset} Dataset:")
    dt_custom = dataset_results[dataset_results['Model'] == 'Custom Decision Tree']['Accuracy'].values[0]
    rf_custom = dataset_results[dataset_results['Model'] == 'Custom Random Forest']['Accuracy'].values[0]
    et_custom = dataset_results[dataset_results['Model'] == 'Custom Extra Trees']['Accuracy'].values[0]

    print(f"  Decision Tree Accuracy: {dt_custom:.4f}")
    print(f"  Random Forest Accuracy: {rf_custom:.4f} (Improvement: {(rf_custom-dt_custom)*100:.2f}%)")
    print(f"  Extra Trees Accuracy:   {et_custom:.4f} (Improvement: {(et_custom-dt_custom)*100:.2f}%)")

# Analysis 2: Custom vs Sklearn
print("\n\n2. CUSTOM IMPLEMENTATION vs SCIKIT-LEARN:")
print("-" * 80)
model_types = ['Decision Tree', 'Random Forest', 'Extra Trees']
for model_type in model_types:
    print(f"\n{model_type}:")
    for dataset in ['Iris', 'Wine']:
        custom_model = f'Custom {model_type}'
        sklearn_model = f'Sklearn {model_type}'

        custom_acc = results_df[(results_df['Model'] == custom_model) &
                                (results_df['Dataset'] == dataset)]['Accuracy'].values[0]
        sklearn_acc = results_df[(results_df['Model'] == sklearn_model) &
                                 (results_df['Dataset'] == dataset)]['Accuracy'].values[0]

        diff = (custom_acc - sklearn_acc) * 100
        print(f"  {dataset}: Custom={custom_acc:.4f}, Sklearn={sklearn_acc:.4f}, Diff={diff:.2f}%")

# Analysis 3: Best models per dataset
print("\n\n3. BEST PERFORMING MODELS:")
print("-" * 80)
for dataset in ['Iris', 'Wine']:
    dataset_results = results_df[results_df['Dataset'] == dataset].copy()
    best_idx = dataset_results['Accuracy'].idxmax()
    best_model = dataset_results.loc[best_idx]
    print(f"\n{dataset} Dataset:")
    print(f"  Model: {best_model['Model']}")
    print(f"  Accuracy: {best_model['Accuracy']:.4f}")
    print(f"  F1-Score: {best_model['F1-Score']:.4f}")
    print(f"  AUROC: {best_model['AUROC']:.4f}")

print("\n" + "=" * 80)


KEY FINDINGS AND ANALYSIS

1. ENSEMBLE METHODS vs DECISION TREES:
--------------------------------------------------------------------------------

Iris Dataset:
  Decision Tree Accuracy: 0.8667
  Random Forest Accuracy: 0.9111 (Improvement: 4.44%)
  Extra Trees Accuracy:   0.9333 (Improvement: 6.67%)

Wine Dataset:
  Decision Tree Accuracy: 0.8889
  Random Forest Accuracy: 0.9815 (Improvement: 9.26%)
  Extra Trees Accuracy:   0.9815 (Improvement: 9.26%)


2. CUSTOM IMPLEMENTATION vs SCIKIT-LEARN:
--------------------------------------------------------------------------------

Decision Tree:
  Iris: Custom=0.8667, Sklearn=0.8667, Diff=0.00%
  Wine: Custom=0.8889, Sklearn=0.9074, Diff=-1.85%

Random Forest:
  Iris: Custom=0.9111, Sklearn=0.9556, Diff=-4.44%
  Wine: Custom=0.9815, Sklearn=0.9630, Diff=1.85%

Extra Trees:
  Iris: Custom=0.9333, Sklearn=0.9333, Diff=0.00%
  Wine: Custom=0.9815, Sklearn=0.9815, Diff=0.00%


3. BEST PERFORMING MODELS:
--------------------------------------

## 10. Conclusion

This assignment successfully implemented three tree-based learning algorithms from scratch:

1. **Decision Tree**: Basic implementation using Gini impurity/entropy for split selection
2. **Random Forest**: Ensemble method using bootstrap sampling and random feature selection
3. **Extra Trees**: Ensemble method with random threshold selection and no bootstrap

### Key Observations:

- **Ensemble superiority**: Random Forest and Extra Trees generally outperform single Decision Trees due to variance reduction
- **Custom vs Sklearn**: Our implementations achieve comparable performance to scikit-learn, validating correctness
- **Bias-Variance Tradeoff**: Ensemble methods effectively reduce overfitting while maintaining low bias

### Implementation Features:
-  Configurable hyperparameters (max_depth, min_samples_split, n_estimators, max_features)
-  Support for classification tasks
-  Comprehensive evaluation metrics (Accuracy, F1-Score, AUROC)
-  Reproducible results with fixed random seed
-  Clean, modular, and well-documented code
